In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from openai import OpenAI 
openai_client = OpenAI()

#### A first query

In [3]:
def llm(prompt): 
    response = openai_client.responses.create(
        model='gpt-5.4-mini', 
        input=prompt
    )
    return response.output_text 

In [ ]:
question = 'I just discovered the course. Can I join now?'
answer = llm(question)
print(answer)

#### With Context 

In [9]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [11]:
prompt = f'''
Your task is to answer questions from the course participants based on the provided context. 

Use the context to find relevant information and provide accurate answers. If the answer is 
not found in the context, respond with "I don't know".

Question: 
{question} 

Context: 
{context}
'''

In [15]:
print(prompt)


Your task is to answer questions from the course participants based on the provided context. 

Use the context to find relevant information and provide accurate answers. If the answer is 
not found in the context, respond with "I don't know".

Question: 
I just discovered the course. Can I join now? 

Context: 

I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students pa

In [16]:
question = 'I just discovered the course. Can I join now?'
answer = llm(prompt)
print(answer)

Yes, you can still join. If you want to receive a certificate, make sure to submit your project while submissions are still open.


#### RAG 

In [4]:
def rag(question): 
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [5]:
import requests 

docs_url = 'https://datatalks.club/faq/json/courses.json'
response = requests.get(docs_url)
courses_raw = response.json()

In [6]:
documents = []

url_prefix = 'https://datatalks.club/faq'

for course in courses_raw: 
    course_url = f'{url_prefix}{course["path"]}'
    course_response = requests.get(course_url)
    course_data = course_response.json() 

    documents.extend(course_data) 
len(documents)

1342

#### Search Engine 

Typical ones: Lucene, ElasticSearch, Solr. For small datasets such as those in this lessons, we will use `minsearch`. 

In [ ]:
from minsearch import Index 

index = Index(
    text_fields=['question', 'section', 'answer'],  # fields you can use to perform search 
    keyword_fields=['course']   # used to restritch search space to a specific sub-space 
)

index.fit(documents)

In [30]:
search_results = index.search(question, filter_dict={'course': 'llm-zoomcamp'}, num_results=5)

In [37]:
def search(question): 
    boost_dict={'question': 2.0, 'section': 0.5, 'answer': 1.0} # level of importance (1.0 by default)
    filter_dict={'course': 'llm-zoomcamp'}

    return index.search(
        question, 
        boost_dict=boost_dict,
        filter_dict=filter_dict,     
        num_results=5,
    )

In [38]:
search_results = search(question)

#### Prompt Building

In [90]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants based on the provided context. 

Use the context to find relevant information and provide accurate answers. If the answer is 
not found in the context, respond with "I don't know".
'''

In [91]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [69]:
def build_context(search_results):
    lines = []

    for doc in search_results: 
        lines.append(doc['section']) 
        lines.append('Q:' + doc['question']) 
        lines.append('A:' + doc['answer'])
        lines.append('')

    return '\n'.join(lines).strip()

In [70]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question, 
        context=context
    )
    return prompt.strip() 

In [71]:
prompt = build_prompt(question, search_results)
print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q:I just discovered the course. Can I still join?
A:Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q:Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A:You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q:Certificate: Can I follow the course in a self-paced mode and get a certificate?
A:No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You

#### LLM 

In [74]:
response = openai_client.responses.create(
    model='gpt-5.4-mini', 
    input=prompt
)

In [85]:
print(response.model_dump_json(indent=2))

{
  "id": "resp_0e5af8b4ef8729d1006a2c2b42962c819fa4dfbafad266c21b",
  "created_at": 1781279554.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-5.4-mini-2026-03-17",
  "object": "response",
  "output": [
    {
      "id": "msg_0e5af8b4ef8729d1006a2c2b436a04819f961c9057667457f4",
      "content": [
        {
          "annotations": [],
          "text": "Yes — you can still join and start learning now.\n\nIf you want a certificate, make sure you submit your project while submissions are still being accepted.",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message",
      "phase": "final_answer"
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [],
  "top_p": 0.98,
  "background": false,
  "completed_at": 1781279555.0,
  "conversation": null,
  "max_output_tokens": null,
  "max_

In [ ]:
# response.output_text # --> shortcut for: 
response.output[0].content[0].text

'Yes — you can still join and start learning now.\n\nIf you want a certificate, make sure you submit your project while submissions are still being accepted.'

In [83]:
# Check tokens usage 
response.usage

ResponseUsage(input_tokens=332, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=34, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=366)

In [84]:
# you can find price data in the openai dashboard page (specific model's pricing) -- price per million tokens 
input_price = 0.75 / 1_000_000 
output_price = 4.50 / 1_000_000 

cost = (
    response.usage.input_tokens * input_price + 
    response.usage.output_tokens * output_price
)

cost # <- price for the query 


0.000402

In [95]:
message_history = [
    {'role': 'developer', 'content': 'INSTRUCTIONS'},  # constant: role=system also works 
    {'role': 'user', 'content': prompt}   # varies 
]

response = openai_client.responses.create(
    model='gpt-5.4-mini', 
    input=message_history
)

In [96]:
response.output_text

'Yes — you can still join now.\n\nIf you want a certificate, make sure to submit your project while submissions are still open.'

In [97]:
def llm(instructions, user_prompt, model='gpt-5.4-mini'): 
    message_history = [
        {'role': 'developer', 'content': instructions},  
        {'role': 'user', 'content': user_prompt}  
    ]

    response = openai_client.responses.create(
        model='gpt-5.4-mini', 
        input=message_history
    )

    return response.output_text 

In [100]:
def rag(query, model='gpt-5.4-mini'): 
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer 

In [101]:
answer = rag(question)
print(answer) 

Yes, you can still join now. If you want to receive a certificate, you need to submit your project while submissions are still being accepted.
